Pada tahap ini, kita memuat dataset Social Battery dan membagi pengguna secara acak (50:50) ke dalam dua kelompok eksperimen:
* **Origin (Kontrol):** Pengguna tidak mendapat *push notification*.
* **Varian (Perlakuan):** Pengguna mendapat *push notification* pengingat.

Untuk keperluan simulasi, kita mengasumsikan kelompok Varian mencatat sedikit lebih banyak kegiatan karena efek dari notifikasi.

Sebelum melakukan pengujian statistik menggunakan T-Test, kita perlu mendefinisikan hipotesis eksperimen dan batas toleransi kesalahannya:

**1. Rumusan Hipotesis**
* **Hipotesis Nol (H0):** Tidak ada perbedaan rata-rata jumlah kegiatan yang dicatat antara kelompok yang mendapat notifikasi dan kelompok yang tidak mendapat notifikasi (Notifikasi tidak memberikan efek).
* **Hipotesis Alternatif (H1):** Terdapat perbedaan rata-rata jumlah kegiatan yang dicatat antara kelompok yang mendapat notifikasi dan kelompok yang tidak mendapat notifikasi (Notifikasi memberikan efek).

**2. Taraf Signifikansi (Alpha)**
* **Nilai Alpha = 0.05** (Tingkat signifikansi 5%).
* Artinya, kita menggunakan tingkat kepercayaan 95% dan bersedia menerima risiko kesalahan sebesar 5% dalam mengambil kesimpulan.
* **Kriteria Pengambilan Keputusan:**
  * Jika **p-value < 0.05**, maka H0 ditolak (Notifikasi terbukti berdampak signifikan).
  * Jika **p-value >= 0.05**, maka H0 gagal ditolak (Notifikasi tidak terbukti berdampak signifikan).

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

df_raw = pd.read_csv('Social Battre.csv')

df = df_raw[['user_id', 'date', 'total_events', 'total_duration_minutes']].dropna().copy()

np.random.seed(42)

# Membagi user ke dalam dua kelompok eksperimen (50:50)
df['user_type'] = np.random.choice(['origin', 'varian'], size=len(df), p=[0.5, 0.5])

# Simulasi: Memberikan efek notifikasi pada kelompok Varian
def apply_notification_effect(row):
    if row['user_type'] == 'varian':
        # Asumsi: Mendapat notifikasi membuat user mencatat 1-2 kegiatan ekstra
        return row['total_events'] + np.random.randint(1, 3)
    return row['total_events']

df['total_events'] = df.apply(apply_notification_effect, axis=1)

df.head()

,user_id,date,total_events,total_duration_minutes,user_type
0,hafizhan shidqi_1,2026-01-01,5,918.0,origin
1,gandhi wibowo_2,2026-01-01,4,960.0,varian
2,aldio mahendra purwandrarto_3,2026-01-01,4,888.0,varian
3,benny putra_4,2026-01-01,7,1008.0,varian
4,vicky vernando dasta_5,2026-01-02,2,1050.0,origin


Kita akan membandingkan rata-rata metrik `total_events` antara kelompok Origin dan Varian. Uji statistik yang digunakan adalah *Independent Two-Sample T-Test* dengan asumsi varians kedua kelompok tidak sama.

In [ ]:
data_tanpa_notif = df[df['user_type'] == 'origin']['total_events']
data_dengan_notif = df[df['user_type'] == 'varian']['total_events']

print("=== RINGKASAN DATA (TOTAL EVENTS) ===")
print(f"Kelompok Origin (Tanpa Notif) : {data_tanpa_notif.mean():.2f} kegiatan/hari (n={len(data_tanpa_notif)})")
print(f"Kelompok Varian (Dapat Notif) : {data_dengan_notif.mean():.2f} kegiatan/hari (n={len(data_dengan_notif)})\n")

# Independent Two-Sample T-Test
alpha = 0.05
t_statistic, p_value = stats.ttest_ind(data_tanpa_notif, data_dengan_notif, equal_var=False)

print("=== HASIL UJI STATISTIK ===")
print(f"Nilai T-Statistic : {t_statistic:.4f}")
print(f"Nilai P-Value     : {p_value:.10f}\n")

=== RINGKASAN DATA (TOTAL EVENTS) ===
Kelompok Origin (Tanpa Notif) : 5.91 kegiatan/hari (n=4203)
Kelompok Varian (Dapat Notif) : 7.37 kegiatan/hari (n=4097)

=== HASIL UJI STATISTIK ===
Nilai T-Statistic : -19.3145
Nilai P-Value     : 0.0000000000



Pada tahap akhir, kita mengevaluasi nilai *p-value* dibandingkan dengan tingkat signifikansi (*alpha* = 0.05) untuk menentukan apakah Hipotesis Nol (H0) ditolak atau tidak.

In [ ]:
print("=== KESIMPULAN ===")
if p_value < alpha:
    print(f"HIPOTESIS NOL (H0) DITOLAK (p-value < {alpha}).")
    print("Terdapat perbedaan yang signifikan secara statistik pada jumlah kegiatan yang dicatat.")
    print("-> Rekomendasi: Mengirimkan Push Notification terbukti efektif meningkatkan aktivitas pengguna dalam mencatat jadwal. Fitur notifikasi harian ini layak diimplementasikan secara permanen ke seluruh pengguna aplikasi CALM.")
else:
    print(f"HIPOTESIS NOL (H0) GAGAL DITOLAK (p-value >= {alpha}).")
    print("Tidak ada perbedaan yang signifikan secara statistik antara kelompok yang diberi notifikasi dan yang tidak.")
    print("-> Rekomendasi: Push Notification tidak memberikan dampak yang berarti. Tim perlu mengevaluasi ulang isi pesan notifikasi atau mencoba strategi lain.")

=== KESIMPULAN ===
HIPOTESIS NOL (H0) DITOLAK (p-value < 0.05).
Terdapat perbedaan yang signifikan secara statistik pada jumlah kegiatan yang dicatat.
-> Rekomendasi: Mengirimkan Push Notification terbukti efektif meningkatkan aktivitas pengguna dalam mencatat jadwal. Fitur notifikasi harian ini layak diimplementasikan secara permanen ke seluruh pengguna aplikasi CALM.
